# Project - Airline AI Assistant

We'll now bring together what we've learned to make an AI Customer Support assistant for an Airline

In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..','..')))
from ai_tools import LLMQuery

In [ ]:
# Initialization

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4.1-mini"
openai = OpenAI()

# As an alternative, if you'd like to use Ollama instead of OpenAI
# Check that Ollama is running for you locally (see week1/day2 exercise) then uncomment these next 2 lines
# MODEL = "llama3.2"
# openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')


In [2]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()

## Tools

Tools are an incredibly powerful feature provided by the frontier LLMs.

With tools, you can write a function, and have the LLM call that function as part of its response.

Sounds almost spooky.. we're giving it the power to run code on our machine?

Well, kinda.

In [10]:
# Let's start by making a useful function

ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f"Tool called for city {destination_city}")
    price = ticket_prices.get(destination_city.lower(), "Unknown ticket price")
    return f"The price of a ticket to {destination_city} is {price}"


In [25]:
get_ticket_price("London")

Tool called for city London


'The price of a ticket to London is $799'

In [3]:
# There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

set_price_function = {
    "name": "set_ticket_price",
    "description": "Set the price of a ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to set the price for.",
            },
            "price": {
                "type": "integer",
                "description": "The price of the ticket",
            },
        },
        "required": ["destination_city", "price"],
        "additionalProperties": False
    }
}

In [4]:
# And this is included in a list of tools:

tools = [{"type": "function", "function": price_function}, {"type": "function", "function": set_price_function}]

In [ ]:
tools

## Getting OpenAI to use our Tool

There's some fiddly stuff to allow OpenAI "to call our tool"

What we actually do is give the LLM the opportunity to inform us that it wants us to run the tool.

Here's how the new chat function looks:

In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        response = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        response = openai.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [21]:
# We have to write that function handle_tool_call:

def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    if tool_call.function.name == "get_ticket_price":
        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get('destination_city')
        price_details = get_ticket_price(city)
        response = {
            "role": "tool",
            "content": price_details,
            "tool_call_id": tool_call.id
        }
    return response

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

In [ ]:
client = LLMQuery(system_prompt=system_message, tools=tools, model="gpt-5.2")

def chat(message, _):
    response = client.query(message)
    print(response)
    while client.tool_calls:
        tool_response = handle_tool_call_llm(client.tool_calls)
        client.append_tool_result(tool_response)
        response = client.query()
    return response
        

In [36]:
# We have to write that function handle_tool_call:

def handle_tool_call_llm(tool_calls):
    tool_response = []
    for tool_call in tool_calls:
        if tool_call['function']['name'] == "get_ticket_price":
            arguments = json.loads(tool_call['function']['arguments'])
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            tool_response.append({"tool_call_id": tool_call['id'], "output": price_details})
            print(tool_response)
        if tool_call['function']['name'] == "set_ticket_price":
            arguments = json.loads(tool_call['function']['arguments'])
            city = arguments.get('destination_city')
            price = arguments.get('price')
            details = set_ticket_price(city, price)   
            tool_response.append({"tool_call_id": tool_call['id'], "output": details})
            print(tool_response)
    return tool_response

In [ ]:
def handle_tool_call_llm(tool_calls):
    tool_response = []
    for tool_call in tool_calls:
        function_name = tool_call['function']['name']
        arguments = json.loads(tool_call['function']['arguments'])
        
        # Dynamically find and call the function by name
        if function_name in globals():
            function_to_call = globals()[function_name]
            result = function_to_call(**arguments)
            tool_response.append({"tool_call_id": tool_call['id'], "output": result})
            print(tool_response)
        else:
            print(f"Warning: Function {function_name} not found")
            
    return tool_response

In [37]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.



DATABASE TOOL CALLED: Getting price for London
[{'tool_call_id': 'call_MupeM5O85zBNld7W1lSeNfOw', 'output': 'Ticket price to London is $799.0'}]

DATABASE TOOL CALLED: Getting price for Munich
[{'tool_call_id': 'call_H2Y1tQnerjZw57nQfGIvMjUQ', 'output': 'Ticket price to Munich is $1.0'}]
DATABASE TOOL CALLED: Getting price for Berlin
[{'tool_call_id': 'call_H2Y1tQnerjZw57nQfGIvMjUQ', 'output': 'Ticket price to Munich is $1.0'}, {'tool_call_id': 'call_hHo8SFK6x757fnSthgNF2Xto', 'output': 'No price data available for this city'}]

[{'tool_call_id': 'call_ZbZYW2XbyQEG6qZH6qzXnxVJ', 'output': 'Ticket price for Berlin set to 1'}]

DATABASE TOOL CALLED: Getting price for Munich
[{'tool_call_id': 'call_yDmNy5hcg62xUz9USI0QxOyr', 'output': 'Ticket price to Munich is $1.0'}]
DATABASE TOOL CALLED: Getting price for Berlin
[{'tool_call_id': 'call_yDmNy5hcg62xUz9USI0QxOyr', 'output': 'Ticket price to Munich is $1.0'}, {'tool_call_id': 'call_ypyiGtq5gBsN4xSR4ERQtuif', 'output': 'Ticket price to Be

Whats the Price to London? Only if this is under 1000$ query the price for Berlin. Do NOT query the price before you checked the price of London.

In [39]:
client.chat_history

[{'role': 'user', 'content': 'How much is a Ticket to London?'},
 {'role': 'assistant',
  'content': None,
  'tool_calls': [{'id': 'call_MupeM5O85zBNld7W1lSeNfOw',
    'function': {'arguments': '{"destination_city":"London"}',
     'name': 'get_ticket_price'},
    'type': 'function'}]},
 {'role': 'tool',
  'content': 'Ticket price to London is $799.0',
  'tool_call_id': 'call_MupeM5O85zBNld7W1lSeNfOw'},
 {'role': 'assistant', 'content': 'A ticket to London costs $799.'},
 {'role': 'user',
  'content': 'How much is a Ticket to Munich compared to Berlin?'},
 {'role': 'assistant',
  'content': None,
  'tool_calls': [{'id': 'call_H2Y1tQnerjZw57nQfGIvMjUQ',
    'function': {'arguments': '{"destination_city": "Munich"}',
     'name': 'get_ticket_price'},
    'type': 'function'},
   {'id': 'call_hHo8SFK6x757fnSthgNF2Xto',
    'function': {'arguments': '{"destination_city": "Berlin"}',
     'name': 'get_ticket_price'},
    'type': 'function'}]},
 {'role': 'tool',
  'content': 'Ticket price to 

## Let's make a couple of improvements

Handling multiple tool calls in 1 response

Handling multiple tool calls 1 after another

In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [29]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
        if tool_call.function.name == "set_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price = arguments.get('price')
            price_set = set_ticket_price(city, price)
            responses.append({
                "role": "tool",
                "content": price_set,
                "tool_call_id": tool_call.id
            })      
    return responses

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [5]:
import sqlite3


In [6]:
DB = "prices.sqlite"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

In [9]:
def get_ticket_price(destination_city):
    print(f"DATABASE TOOL CALLED: Getting price for {destination_city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (destination_city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {destination_city} is ${result[0]}" if result else "No price data available for this city"

In [8]:
get_ticket_price("Munich")

DATABASE TOOL CALLED: Getting price for Munich


'Ticket price to Munich is $1.0'

In [7]:
def set_ticket_price(destination_city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (destination_city.lower(), price, price))
        conn.commit()
    return f"Ticket price for {destination_city} set to {price}"

In [18]:
ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

In [14]:
get_ticket_price("Vienna")

DATABASE TOOL CALLED: Getting price for Vienna


'Ticket price to Vienna is $1042.0'

DATABASE TOOL CALLED: Getting price for Tokyo
DATABASE TOOL CALLED: Getting price for Vienna


In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

## Exercise

Add a tool to set the price of a ticket!

In [ ]:
client = LLMQuery(system_prompt=system_message, tools=tools, functions=[get_ticket_price, set_ticket_price], model="gemini-3-pro-preview")

def chat(message, _):
    response = client.query(message)
    response = client.get_tool_responses()
    return response

gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


DATABASE TOOL CALLED: Getting price for Munich
DATABASE TOOL CALLED: Getting price for Tokyo
DATABASE TOOL CALLED: Getting price for Berlin
DATABASE TOOL CALLED: Getting price for Paris


In [15]:
client.chat_history

[{'role': 'user',
  'content': 'Set the price to Vienna to the Amount (Munich + Tokyo) - (Berlin + Paris) and mulitply the result by two if the price to Tokyo is more thann $1000. Explain what you did.'},
 {'role': 'assistant',
  'content': None,
  'tool_calls': [{'id': 'function-call-14332845062937940772',
    'function': {'arguments': '{"destination_city":"Munich"}',
     'name': 'get_ticket_price'},
    'type': 'function',
    'extra_content': {'google': {'thought_signature': 'ErQGCrEGAXLI2nyPla6J2R6y8zsoMaXH2m3SDtE1RfNs98nwxjSK4y7VLiDzJ7PGM3fH93zS8CCE4m7dIlU/pfT2eUUjPZ3NfSbkM+z0aDt8eQTwSrvmKCMlc3Rno+t6wrwt97LIm1oxBIdCky0CxhOMcyeg+a5J9/4ALud7mtz6YjHtXt5c+YtTY517Vscg7jhxiM9vt4L4sqjwXaDMk6hOSJzsdXOcXnEwp+yJMOAJSJ0vONR72i+KAdFXCPDlRdpeYpfwOoqCophqv3nTFq4esMGg0nTf8JuRejAle8naalJ0+XWJqzHqQF27Jae+mnC1WGiVk0ieDc8FcOpBPIMnYSCdd8pYvYYM+N3WiLOiY+7bvNCTEaFzfQEmhVznEIck8ZERZ5lMhLcuaxzKoXIx+CeASv3n1+/MSwGNpyF+9yKMWYtAb4U7+dVb5dRiCTAX6aUEoXmgfpEr7IYb130nSM7XbZS/Dve0iCWfZxsy3B1kzr/N4bNmwVgH6q2JcTN

DATABASE TOOL CALLED: Getting price for Edinburgh


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business Applications</h2>
            <span style="color:#181;">Hopefully this hardly needs to be stated! You now have the ability to give actions to your LLMs. This Airline Assistant can now do more than answer questions - it could interact with booking APIs to make bookings!</span>
        </td>
    </tr>
</table>